# Set up

### Imports

In [ ]:
import pandas as pd



### Load Data

In [ ]:
#df = pd.read_sas("data/LLCP2025XPT/LLCP2025.XPT", format="xport", encoding="latin-1")
#df.to_parquet("brfss2025.parquet")
#df = pd.read_parquet("../data/brfss2025.parquet")
#df_il = df[df["_STATE"] == 17]
#df_il.to_parquet("../data/brfss2025_il.parquet")

In [ ]:
target = "DIABETE4"
filter_cols = ["DIABTYPE"]        # used only to drop type 1, never as a predictor

cat_predictors = [
    "GENHLTH", "_PHYS14D", "_MENT14D",
    "_HLTHPL2", "_TOTINDA",
    "_RFHYPE6", "_RFCHOL3", "_MICHD", "_ASTHMS1", "_DRDXAR2",
    "_SEX", "_EDUCAG", "_INCOMG1", "_RACE",
    "_SMOKER3", "_CURECI3", "_RFBING6",
]
num_predictors = [
    "_AGE80", "HTIN4", "_BMI5", "_CHLDCNT", "_DRNKWK3",
    "FTJUDA2_", "FRUTDA2_", "GRENDA1_", "FRNCHDA_", "VEGEDA2_",
]
predictors = cat_predictors + num_predictors
assert len(set(predictors)) == len(predictors), "duplicate column in list"

df_raw = pd.read_parquet(
    "../data/brfss2025_il.parquet",
    columns=[target] + filter_cols + predictors,
)

df = df_raw[df_raw["DIABETE4"].isin([1, 2, 3, 4]) & (df_raw["DIABTYPE"] != 1)].copy()
df["DIABETES"] = (df["DIABETE4"] == 1).astype(int)

Column Notes:
- _BMI5, _DRNKWK3 AND FRUIT/VEG cols need /100
- _RFCHOL3 has missing values for people who weren't asked the question because they had never had cholesterol checked, code appropriately
- _CHLDCNT needs -1 to represent actualcount, 9 is NaN, fix zero?

In [ ]:
display(df["DIABETES"].value_counts(dropna=False))

d = df_raw[df_raw["DIABETE4"].isin([1, 2, 3, 4])]
display(d.loc[d["DIABETE4"] == 1, "DIABTYPE"].value_counts(dropna=False))

### Missingness

In [ ]:
missing = (df.isna().mean().sort_values() * 100).round(2)   # Percentage of missing values for each column
missing[predictors + ["DIABETES"]].sort_values()